# SHAP Feature Interpretation for Student Performance Prediction

## Overview
This notebook demonstrates how to use **SHAP (SHapley Additive exPlanations)** to interpret and explain which features are most crucial in predicting whether students pass or fail.

### What is SHAP?
SHAP is a game-theoretic approach based on Shapley values that provides a unified measure of feature importance. It answers the question: **"How much did each feature contribute to this specific prediction?"**

### Why SHAP Matters for Student Performance:
- 🎯 **Feature Attribution**: Understand which student characteristics drive pass/fail predictions
- 👤 **Individual Explanations**: Explain why a specific student is predicted to pass or fail  
- 📊 **Global Insights**: Identify the most important factors across all students
- 🔍 **Actionable Insights**: Help educators target interventions based on key predictive factors

## Section 1: Install and Import SHAP Libraries

We'll start by installing SHAP and importing all necessary packages for data handling, model training, and visualization.

In [ ]:
# Install SHAP library
import subprocess
import sys

try:
    import shap
    print("✓ SHAP is already installed")
except ImportError:
    print("Installing SHAP...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    print("✓ SHAP installed successfully")

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set visualization styles
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")
print(f"✓ SHAP version: {shap.__version__}")

## Section 2: Load and Prepare Student Data

Load the student dataset, perform exploratory analysis, encode categorical features, and prepare data for model training.

In [ ]:
# Load the student dataset
df = pd.read_csv("../data/student-mat.csv", sep=";")
print(f"✓ Dataset loaded: {df.shape[0]} students, {df.shape[1]} features")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Exploratory Data Analysis
print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nBasic Statistics:\n{df.describe()}")

In [ ]:
# Data Preprocessing
df_processed = df.copy()

# Drop unnecessary columns
if "school" in df_processed.columns:
    df_processed = df_processed.drop(columns=["school"])

# Identify categorical columns
categorical_cols = df_processed.select_dtypes(include="object").columns.tolist()
numerical_cols = df_processed.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Encode categorical variables
le = LabelEncoder()
for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))

print(f"\n✓ All categorical variables encoded")
print(f"Processed dataset shape: {df_processed.shape}")

In [ ]:
# Create target variable (Pass/Fail based on G3 - final grade)
# We'll define Pass as G3 >= 10, Fail as G3 < 10
threshold = 10
df_processed['target'] = (df_processed['G3'] >= threshold).astype(int)

print(f"Target Variable Distribution:")
print(df_processed['target'].value_counts())
print(f"\nPass Rate: {df_processed['target'].mean()*100:.2f}%")
print(f"Fail Rate: {(1-df_processed['target'].mean())*100:.2f}%")

# Prepare features and labels
X = df_processed.drop(columns=['G1', 'G2', 'G3', 'target'])
y = df_processed['target']

feature_names = X.columns.tolist()
print(f"\n✓ Features: {len(feature_names)}")
print(f"✓ Target: Binary (0=Fail, 1=Pass)")

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Data scaled and split:")
print(f"  Training set: {X_train.shape}")
print(f"  Test set: {X_test.shape}")
print(f"  Train pass rate: {y_train.mean()*100:.2f}%")
print(f"  Test pass rate: {y_test.mean()*100:.2f}%")

## Section 3: Train a Classification Model

Build and train a Random Forest classifier to predict student pass/fail outcomes.

In [ ]:
# Train Random Forest Classifier
print("Training Random Forest Classifier...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("✓ Model training complete")

# Evaluate model
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"\nModel Performance:")
print(f"  Training Accuracy: {train_accuracy:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test, target_names=['Fail', 'Pass']))

## Section 4: Initialize SHAP Explainer

Create a SHAP TreeExplainer for our Random Forest model. This will calculate how each feature contributes to predictions.

In [ ]:
### SHAP Explainer Types:
# 1. **TreeExplainer**: For tree-based models (RandomForest, XGBoost, etc.) - FAST
# 2. **KernelExplainer**: Model-agnostic but slower - works for any model
# 3. **LinearExplainer**: For linear models
# 4. **DeepExplainer**: For deep neural networks

print("Initializing SHAP TreeExplainer...")
print("TreeExplainer is ideal for Random Forest models - it's fast and accurate!\n")

# Create TreeExplainer using training data as background
explainer = shap.TreeExplainer(model)

print("✓ SHAP TreeExplainer initialized")
print(f"✓ Base value (expected model output): {explainer.expected_value:.4f}")
print("\nNote: Base value represents the average prediction across training data")

## Section 5: Generate SHAP Values

Calculate SHAP values for all predictions. This quantifies each feature's contribution to the model's output.

In [ ]:
print("Computing SHAP values for test set...")
print("This may take a minute depending on model complexity...\n")

# Calculate SHAP values for the test set
# For classification, we get a 3D array: [sample, feature, class]
shap_values = explainer.shap_values(X_test)

print("✓ SHAP values computed successfully!")
print(f"✓ Shape of SHAP values: {np.array(shap_values).shape}")
print(f"✓ Number of samples: {X_test.shape[0]}")
print(f"✓ Number of features: {X_test.shape[1]}")
print(f"✓ Number of classes: {len(shap_values)}")
print(f"\nNotes:")
print(f"  - shap_values[0]: SHAP values for Fail predictions")
print(f"  - shap_values[1]: SHAP values for Pass predictions")

## Section 6: Visualize Feature Importance with Summary Plot

Create summary plots to identify which features are most important in determining pass/fail predictions.

### Understanding the SHAP Summary Plot:
- **Bar Plot**: Shows mean absolute SHAP values - overall feature importance
- **Beeswarm Plot**: Shows individual SHAP values colored by feature value
  - Red/Blue colors represent high/low feature values
  - Position shows impact on model output
  - Points spreading right = higher positive impact on predictions

In [ ]:
# SHAP Bar Plot - Feature Importance
print("Generating SHAP Bar Plot (Mean Absolute Feature Importance)...\n")

plt.figure(figsize=(12, 8))
# Use SHAP values for Pass class (class 1)
shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, 
                  plot_type="bar", show=False)
plt.title("SHAP Feature Importance for Student Pass Predictions\n(Mean |SHAP values|)", 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel("Mean |SHAP value|", fontsize=12)
plt.tight_layout()
plt.show()

print("✓ Bar plot shows which features have the largest impact on predictions")

In [ ]:
# SHAP Beeswarm Plot - Detailed Feature Impact
print("Generating SHAP Beeswarm Plot (Detailed Feature Impact)...\n")

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm Plot - How Features Impact Pass Predictions\n(Each dot = one student)", 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel("SHAP value (impact on model output)", fontsize=12)
plt.tight_layout()
plt.show()

print("✓ Beeswarm plot interpretation:")
print("  - Each dot represents a student")
print("  - Left side = pushes prediction toward FAIL")
print("  - Right side = pushes prediction toward PASS")
print("  - Red = high feature value")
print("  - Blue = low feature value")

## Section 7: Analyze Individual Predictions with Force Plot

Generate force plots for individual students to see how specific feature values pushed their predictions toward pass or fail.

In [ ]:
# Force Plot for Individual Prediction
print("Force Plot Example: Explaining Individual Student Predictions\n")

# Select a student to explain
sample_idx = 0
actual_pred = y_test.iloc[sample_idx]
model_pred_proba = model.predict_proba(X_test)[sample_idx]

print(f"Student #{sample_idx}:")
print(f"  Actual: {'PASS' if actual_pred == 1 else 'FAIL'}")
print(f"  Model Prediction: {'PASS' if y_pred_test[sample_idx] == 1 else 'FAIL'}")
print(f"  Confidence: {max(model_pred_proba)*100:.2f}%\n")

# Create force plot
plt.figure(figsize=(14, 5))
shap.force_plot(explainer.expected_value[1], shap_values[1][sample_idx], X_test[sample_idx],
               feature_names=feature_names, matplotlib=True, show=False)
plt.title(f"SHAP Force Plot - Student #{sample_idx} \n(How features pushed toward {'PASS' if y_pred_test[sample_idx] == 1 else 'FAIL'})", 
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Force Plot for Another Student (Fail case)
print("\n" + "="*70 + "\n")

# Find a student predicted to fail
fail_indices = np.where(y_pred_test == 0)[0]
if len(fail_indices) > 0:
    fail_idx = fail_indices[0]
    
    print(f"Student #{fail_idx}:")
    print(f"  Actual: {'PASS' if y_test.iloc[fail_idx] == 1 else 'FAIL'}")
    print(f"  Model Prediction: {'PASS' if y_pred_test[fail_idx] == 1 else 'FAIL'}")
    print(f"  Confidence: {max(model.predict_proba(X_test)[fail_idx])*100:.2f}%\n")
    
    plt.figure(figsize=(14, 5))
    shap.force_plot(explainer.expected_value[1], shap_values[1][fail_idx], X_test[fail_idx],
                   feature_names=feature_names, matplotlib=True, show=False)
    plt.title(f"SHAP Force Plot - Student #{fail_idx} \n(How features pushed toward FAIL prediction)", 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Section 8: Interpret Feature Contributions with Dependence Plot

Create dependence plots to show the relationship between feature values and their SHAP contributions.

In [ ]:
### Understanding Dependence Plots:
# - X-axis: Feature value
# - Y-axis: SHAP value (impact on prediction)
# - Colored by: Interacting feature value
# - Shows non-linear relationships between features and predictions

# Get top 3 most important features
mean_shap_values = np.abs(shap_values[1]).mean(axis=0)
top_features_idx = np.argsort(mean_shap_values)[-3:]

print("Top 3 Most Important Features:")
for idx, feat_idx in enumerate(top_features_idx[::-1], 1):
    print(f"  {idx}. {feature_names[feat_idx]}")

# Create dependence plots for top features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("SHAP Dependence Plots - Top 3 Features", fontsize=14, fontweight='bold')

for idx, (ax, feat_idx) in enumerate(zip(axes, top_features_idx[::-1])):
    plt.sca(ax)
    shap.dependence_plot(feat_idx, shap_values[1], X_test, 
                        feature_names=feature_names, show=False, ax=ax)
    ax.set_title(f"{feature_names[feat_idx]}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Dependence plots show how feature values relate to their SHAP contributions")

## Section 9: Create Waterfall Plot for Decision Explanation

Build waterfall plots to explain individual predictions step-by-step, showing how each feature pushes the prediction away from the base value.

In [ ]:
### Understanding Waterfall Plots:
# - Base value: Starting point (average prediction)
# - Red bars: Features pushing prediction toward 0 (FAIL)
# - Blue bars: Features pushing prediction toward 1 (PASS)
# - Final bar: Model's actual prediction

print("Creating Waterfall Plots for Individual Predictions\n")

# Waterfall plot for Pass prediction
pass_indices = np.where(y_pred_test == 1)[0]
if len(pass_indices) > 0:
    pass_idx = pass_indices[0]
    
    # Create Explainer object for waterfall plot
    explainer_object = shap.Explainer(model, X_train)
    shap_vals = explainer_object(X_test)
    
    plt.figure(figsize=(12, 6))
    shap.plots.waterfall(shap_vals[pass_idx], show=False)
    plt.title(f"SHAP Waterfall Plot - Student #{pass_idx} Predicted to PASS", 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Waterfall plot for PASS prediction (Student #{pass_idx})")
    print("  - Shows cumulative feature contributions")
    print("  - Base value + contributions = Final prediction")

In [ ]:
# Waterfall plot for Fail prediction
print("\n" + "="*70 + "\n")

if len(fail_indices) > 0:
    fail_idx = fail_indices[0]
    
    plt.figure(figsize=(12, 6))
    shap.plots.waterfall(shap_vals[fail_idx], show=False)
    plt.title(f"SHAP Waterfall Plot - Student #{fail_idx} Predicted to FAIL", 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Waterfall plot for FAIL prediction (Student #{fail_idx})")
    print("  - Red bars show features pushing toward FAIL")
    print("  - Blue bars show features pushing toward PASS (but outweighed)")

## Summary & Key Insights

### What We've Learned:

1. **Feature Importance** 
   - Identified which student characteristics have the largest impact on pass/fail predictions
   - Some features consistently push toward pass, others toward fail

2. **Individual Explanations**
   - Force plots show how specific feature values influenced individual student predictions
   - Understand why a student is predicted to pass or fail

3. **Non-Linear Relationships**
   - Dependence plots reveal how feature values relate to their impact on predictions
   - Shows interactions between features

4. **Prediction Breakdown**
   - Waterfall plots provide step-by-step explanation of predictions
   - See exactly how the model arrived at its decision

### Practical Applications:

✅ **Student Support**: Target interventions based on features pushing students toward fail  
✅ **Policy Making**: Understand which factors impact student success  
✅ **Model Trust**: Build confidence in model decisions with explainability  
✅ **Fair Decision Making**: Ensure important features are driving predictions  

### Next Steps:

1. Use insights to develop targeted intervention programs
2. Monitor changes in feature importance over time
3. Compare SHAP values across different demographic groups
4. Validate model fairness using SHAP explanations

## References & Resources

### SHAP Documentation:
- **Official SHAP GitHub**: https://github.com/slundberg/shap
- **SHAP Documentation**: https://shap.readthedocs.io/
- **SHAP Paper**: "A Unified Approach to Interpreting Model Predictions" (Lundberg & Lee, 2017)

### Key Concepts:
- **Shapley Values**: Game-theoretic concept for fairly distributing feature importance
- **TreeExplainer**: Specialized algorithm for tree-based models (faster than general methods)
- **Feature Attribution**: How much each feature contributed to a specific prediction

### Types of SHAP Plots:
| Plot Type | Purpose | Best For |
|-----------|---------|----------|
| **Summary (Bar)** | Global feature importance | Quick overview of all features |
| **Summary (Beeswarm)** | Detailed feature impact | Individual feature contributions |
| **Force Plot** | Individual prediction | Explaining single student's prediction |
| **Dependence Plot** | Feature relationships | Understanding feature interactions |
| **Waterfall Plot** | Prediction breakdown | Step-by-step explanation |

### Model Explainability Best Practices:
1. Always validate that explanations make domain sense
2. Use multiple explanation methods for robustness
3. Consider feature interactions and correlations
4. Communicate limitations of explanations to stakeholders
5. Monitor explanations over time for consistency